# DATA 266 - Homework 5
## Fine-Tuning an LLM Using LoRA

In this notebook I fine-tune google/flan-t5-small for dialogue summarization on the DialogSum dataset (neil-code/dialogsum-test) using LoRA through the Hugging Face PEFT library.

I used Demo_5_LoRA.ipynb as my starting point and kept its overall flow: load a pretrained model, attach LoRA adapters, check the trainable parameter count, fine-tune, save the adapter, reload it, and run inference. A few things had to change to fit this task though.

| | Demo | This notebook |
| --- | --- | --- |
| Model | Mistral-7B (causal LM) | flan-t5-small (encoder-decoder, seq2seq) |
| Quantization | 8-bit with bitsandbytes | none, the model is only about 77M parameters so it fits easily in fp32 |
| Dataset | English Quotes | DialogSum (dialogue as input, summary as target) |
| LoRA target modules | q_proj, v_proj | q, v (the T5 names for the query and value projections) |
| Trainer / collator | Trainer + DataCollatorForLanguageModeling | Seq2SeqTrainer + DataCollatorForSeq2Seq |
| Generation | sampling (do_sample=True) | greedy decoding (do_sample=False) so before/after outputs are comparable |
| Hugging Face login | required for Mistral | not needed, both the model and dataset are public |

The notebook is organized by task: personal parameters and setup, data preparation, baseline inference, applying LoRA, fine-tuning, post fine-tuning inference, an output comparison, and finally the r=4 vs r=16 rank experiment.

## Personal Parameters

These are the same personal parameters I have used all semester. For this assignment SEED drives reproducibility (Python, NumPy, PyTorch, and the Hugging Face Trainer are all seeded with it), and SLICE is used to pick the two held-out test dialogues for the qualitative before/after comparison. HP_ID, CLS_A, and CLS_B do not map to anything in a summarization task, so they are just reported here for consistency.

In [ ]:
SID4 = 9486
SEED = 9486
SLICE = 486
HP_ID = 0
CLS_A = 6
CLS_B = 0

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")

## Setup: Libraries, Device, and Reproducibility

Unlike the demo, I do not need bitsandbytes or loralib here. flan-t5-small is small enough to train in full precision without quantization, and LoRA itself comes from the peft library. I also skip the Hugging Face login step since everything used is public.

In [ ]:
!pip install -q transformers datasets peft accelerate
# transformers: model, tokenizer, Seq2SeqTrainer and DataCollatorForSeq2Seq
# datasets: loading the DialogSum dataset
# peft: LoRA implementation (LoraConfig, get_peft_model, PeftModel)
# accelerate: required by the Trainer

The device check prints whether the notebook is running on a GPU or CPU. The set_seed function seeds random, NumPy, and PyTorch (including CUDA), and also calls the transformers set_seed helper, so that LoRA weight initialization, dropout, and data shuffling all come out the same across runs.

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU (no CUDA device found)")

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)


set_seed(SEED)

## Task 1: Data Preparation

### Loading and Inspecting DialogSum

DialogSum contains everyday conversations between two or more people, each paired with a short human-written summary. This copy of it (neil-code/dialogsum-test) comes with train, validation, and test splits already defined.

In [ ]:
dataset_name = "neil-code/dialogsum-test"
dataset = load_dataset(dataset_name)  # public dataset, no login needed

print(dataset)
print()
for split_name, split in dataset.items():
    print(f"{split_name:<10} rows = {len(split):>5} | columns = {split.column_names}")

Only two columns matter for this task: dialogue is the input and summary is the target. The id and topic columns are dropped later during tokenization.

### Picking the Evaluation Dialogues and the Training Subset

- **Evaluation dialogues:** The first one is the test dialogue at index SLICE (486). In the DialogSum test split each dialogue shows up in consecutive rows, once per reference summary (ids like test_162_1, test_162_2, test_162_3), so SLICE + 1 would just be the same conversation again. Instead, the second one is the next row after SLICE that has a different dialogue. Since both come from the test split, the model never sees them during training, so the before/after comparison is not affected by data leakage.
- **Training subset:** The full train split has 1,999 examples. I shuffle it with SEED and keep the first 1,000. This is enough for LoRA to learn the DialogSum summary style while keeping each training run short on a Colab T4, which matters because the rank experiment in Task 7 trains a second adapter with the exact same budget.

In [ ]:
test_dialogues = dataset["test"]["dialogue"]
next_distinct = next(i for i in range(SLICE + 1, len(test_dialogues)) if test_dialogues[i] != test_dialogues[SLICE])
EVAL_INDICES = [SLICE, next_distinct]
TRAIN_SIZE = 1000

eval_raw = dataset["test"].select(EVAL_INDICES)
train_raw = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))

assert eval_raw[0]["dialogue"] != eval_raw[1]["dialogue"], "the two evaluation dialogues must be different"

# Leakage check: the evaluation dialogues must not appear in the training subset
train_dialogues = set(train_raw["dialogue"])
for idx, dialogue in zip(EVAL_INDICES, eval_raw["dialogue"]):
    assert dialogue not in train_dialogues, f"test[{idx}] leaked into the training subset"

print("Evaluation dialogues: test indices", EVAL_INDICES, "->", list(eval_raw["id"]))
print("Training subset size:", len(train_raw))
print("No overlap between the training subset and the evaluation dialogues.")

### Converting Each Example into Input and Target

The preprocessing function turns each raw row into an input string and a target string. flan-t5 is an instruction-tuned model, so I wrap the dialogue in a short summarization instruction. This way the baseline model actually knows what task it is being asked to do, and I use the exact same prompt for training and for every inference call afterward.

This is a sequence-to-sequence task, not causal language modeling like in the demo. The dialogue goes into the encoder, and the summary is what the decoder learns to produce. The model is never trained to reproduce the dialogue itself.

In [ ]:
PROMPT_TEMPLATE = "Summarize the following conversation.\n\n{dialogue}\n\nSummary:"


def preprocess_dialogsum(example):
    return {
        "input": PROMPT_TEMPLATE.format(dialogue=example["dialogue"]),
        "target": example["summary"],
    }


train_data = train_raw.map(preprocess_dialogsum)
eval_data = eval_raw.map(preprocess_dialogsum)
print(train_data)

### Tokenizing for Seq2Seq

The input goes through the tokenizer as normal, which gives the encoder input_ids and attention_mask. The target is tokenized with text_target, and those token ids become the labels the decoder is trained on.

Looking at token lengths, most dialogues are around 200 tokens with a long tail past 500, and summaries are usually under 100 tokens. So I truncate inputs at 512 and targets at 128. I do not pad here. DataCollatorForSeq2Seq pads each batch dynamically later and fills the padded label positions with -100, so the loss ignores them.

In [ ]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128


def tokenize_function(batch):
    model_inputs = tokenizer(batch["input"], max_length=MAX_SOURCE_LENGTH, truncation=True)
    labels = tokenizer(text_target=batch["target"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_data.map(tokenize_function, batched=True, remove_columns=train_data.column_names)
print(tokenized_train)

### Two Processed Samples

Printing two processed training samples to confirm that the input and target line up correctly, and that the labels decode back to the summary.

In [ ]:
for i in range(2):
    sample = train_data[i]
    tokens = tokenized_train[i]
    print(f"--- Processed sample {i + 1} (id: {sample['id']}) ---")
    print("INPUT (encoder):")
    print(sample["input"])
    print()
    print("TARGET (decoder labels):")
    print(sample["target"])
    print()
    print("input_ids length:", len(tokens["input_ids"]), "| labels length:", len(tokens["labels"]))
    print("labels decoded  :", tokenizer.decode(tokens["labels"], skip_special_tokens=True))
    print()

## Task 2: Baseline Inference

### Loading flan-t5-small

The base model is loaded in full precision straight onto the device. No quantization config is needed like in the demo, since the whole model is only about 77M parameters.

In [ ]:
set_seed(SEED)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
base_model.eval()

print("Loaded:", model_name)
print(f"Parameters: {sum(p.numel() for p in base_model.parameters()):,}")

### The Two Evaluation Dialogues and Generation Settings

I save the two held-out dialogues in eval_examples so the exact same inputs get reused for the baseline, the r=4 adapter, and the r=16 adapter.

The demo generates with sampling (temperature and top_p), which gives a different output every run. For a before/after comparison I want the output to depend only on the model weights, so I use greedy decoding with do_sample=False, num_beams=1, and a fixed max_new_tokens. The same GEN_KWARGS dictionary is used for every generation in the notebook.

In [ ]:
eval_examples = [
    {
        "example": i + 1,
        "test_index": EVAL_INDICES[i],
        "id": eval_data[i]["id"],
        "dialogue": eval_data[i]["dialogue"],
        "input": eval_data[i]["input"],
        "reference": eval_data[i]["target"],
    }
    for i in range(len(eval_data))
]

GEN_KWARGS = {
    "max_new_tokens": 100,
    "do_sample": False,  # greedy decoding, deterministic
    "num_beams": 1,
}


def generate_summary(model, prompt_text):
    batch = tokenizer(prompt_text, return_tensors="pt", max_length=MAX_SOURCE_LENGTH, truncation=True)
    batch = {k: v.to(model.device) for k, v in batch.items()}
    with torch.no_grad():
        output_tokens = model.generate(**batch, **GEN_KWARGS)
    return tokenizer.decode(output_tokens[0], skip_special_tokens=True).strip()


print("Generation settings:", GEN_KWARGS)

In [ ]:
baseline_outputs = []
for ex in eval_examples:
    baseline_summary = generate_summary(base_model, ex["input"])
    baseline_outputs.append({"example": ex["example"], "id": ex["id"], "baseline": baseline_summary})

    print(f"=============== Example {ex['example']} (test index {ex['test_index']}, id {ex['id']}) ===============")
    print("DIALOGUE:")
    print(ex["dialogue"])
    print()
    print("REFERENCE SUMMARY:")
    print(ex["reference"])
    print()
    print("BASELINE SUMMARY (before fine-tuning):")
    print(baseline_summary)
    print()

## Task 3: Applying LoRA

### Choosing the Target Modules

The demo targets q_proj and v_proj, but those names come from Mistral and do not exist in T5. In T5's attention blocks, the query, key, value, and output projections are just named q, k, v, and o. Before building the LoRA config, I list the linear layer names in the model to confirm q and v actually exist, so PEFT does not silently fail to find them.

In [ ]:
linear_layer_names = sorted({name.split(".")[-1] for name, module in base_model.named_modules()
                             if isinstance(module, torch.nn.Linear)})
print("Linear layer names in flan-t5-small:", linear_layer_names)

TARGET_MODULES = ["q", "v"]
for target in TARGET_MODULES:
    count = sum(1 for name, _ in base_model.named_modules() if name.split(".")[-1] == target)
    assert count > 0, f"target module '{target}' not found"
    print(f"'{target}' found in {count} attention layers")

### Building the LoRA Model

I wrap model loading and get_peft_model in a helper function, since Task 7 needs to build a second LoRA model from a fresh copy of the base model. get_peft_model modifies the model it is given in place, so reusing the same base_model object for both ranks would stack adapters on top of each other. The helper also reseeds right before the LoRA weights are initialized.

The main run uses r=4, lora_alpha=16, and lora_dropout=0.1. task_type is SEQ_2_SEQ_LM instead of the demo's CAUSAL_LM.

In [ ]:
def build_lora_model(r, lora_alpha, lora_dropout):
    set_seed(SEED)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    lora_config = LoraConfig(
        r=r,                              # rank of the low-rank update matrices
        lora_alpha=lora_alpha,            # scaling factor, the update is scaled by lora_alpha / r
        target_modules=TARGET_MODULES,    # T5 query and value projections
        lora_dropout=lora_dropout,        # dropout on the LoRA path
        bias="none",                      # base model biases stay frozen
        task_type=TaskType.SEQ_2_SEQ_LM,  # encoder-decoder model
    )
    return get_peft_model(model, lora_config).to(device)


# Print trainable parameters (same idea as the demo, but also returns the numbers for the tables later)
def print_trainable_parameters(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_pct = 100 * trainable_params / total_params
    print(f"Total parameters     : {total_params:,}")
    print(f"Trainable parameters : {trainable_params:,}")
    print(f"Trainable percentage : {trainable_pct:.4f}%")
    return {"total": total_params, "trainable": trainable_params, "pct": trainable_pct}

In [ ]:
LORA_R4 = {"r": 4, "lora_alpha": 16, "lora_dropout": 0.1}

lora_model_r4 = build_lora_model(**LORA_R4)
params_r4 = print_trainable_parameters(lora_model_r4)

### Why LoRA Only Trains a Small Number of Parameters

LoRA keeps every original weight matrix W frozen and learns a low-rank update instead. For each targeted layer, the new output becomes W·x + (alpha / r)·B·A·x, where A is r × d_in and B is d_out × r. For a 512 × 512 q or v projection in flan-t5-small with r = 4, that is 4 × 512 + 512 × 4 = 4,096 trainable values compared to 262,144 in the full matrix. B starts at zero, so at step 0 the LoRA model behaves exactly like the base model, and training only has to learn the change needed for the new task.

Only the A and B matrices get gradients and optimizer state. Everything else, including the embeddings, the feed-forward layers, the k and o projections, and the LM head, stays frozen. That is why the trainable percentage above is so small. It also means the saved adapter is tiny, and the base model does not need to be copied for each new task.

## Task 4: Fine-Tuning

### Training Setup

A few changes from the demo are needed because this is a seq2seq model:

- **DataCollatorForSeq2Seq instead of DataCollatorForLanguageModeling.** The seq2seq collator pads the encoder inputs and the labels separately, uses -100 for padded label positions so the loss ignores them, and builds the decoder_input_ids from the labels. The language-modeling collator would treat everything as one causal sequence, which is wrong for T5.
- **Seq2SeqTrainer instead of Trainer.**
- **fp16 is off.** T5 models are known to overflow to NaN in fp16, and at 77M parameters fp32 is still fast on a T4.
- **Learning rate of 1e-3.** The demo's 2e-4 is aimed at a 7B model with a 50-step run. A small model with low-rank adapters usually needs a higher learning rate to move noticeably within a short budget.

The training budget is 2 epochs over the 1,000-example subset with a batch size of 8, which comes to 250 optimizer steps. The exact same settings are reused for the r=16 run in Task 7.

In [ ]:
LEARNING_RATE = 1e-3
BATCH_SIZE = 8
NUM_EPOCHS = 2


def make_training_args(output_dir):
    return Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=10,             # short warmup, same idea as the demo
        weight_decay=0.0,
        logging_steps=25,
        save_strategy="no",          # only the final adapter is saved, with save_pretrained below
        fp16=False,                  # T5 is unstable in fp16
        seed=SEED,
        data_seed=SEED,
        report_to="none",
        use_cpu=(device == "cpu"),
    )


def train_lora(lora_model, output_dir):
    args = make_training_args(output_dir)
    trainer = Seq2SeqTrainer(
        model=lora_model,
        args=args,
        train_dataset=tokenized_train,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=lora_model, label_pad_token_id=-100),
        processing_class=tokenizer,
    )
    train_result = trainer.train()
    return trainer, train_result


ADAPTER_DIR_R4 = "outputs/dialogsum_lora_r4"

print("Training configuration")
print(f"  model             : {model_name}")
print(f"  LoRA config       : {LORA_R4}, target_modules={TARGET_MODULES}")
print(f"  training examples : {len(tokenized_train)}")
print(f"  epochs            : {NUM_EPOCHS}")
print(f"  batch size        : {BATCH_SIZE}")
print(f"  steps             : {NUM_EPOCHS * (len(tokenized_train) // BATCH_SIZE + (len(tokenized_train) % BATCH_SIZE > 0))}")
print(f"  learning rate     : {LEARNING_RATE}")
print(f"  seed              : {SEED}")
print(f"  device            : {device}")

In [ ]:
trainer_r4, train_result_r4 = train_lora(lora_model_r4, ADAPTER_DIR_R4)

### Training Logs and Final Loss

In [ ]:
def print_training_log(trainer, train_result):
    for entry in trainer.state.log_history:
        if "loss" in entry:
            print(f"step {entry['step']:>4} | epoch {entry['epoch']:.2f} | loss {entry['loss']:.4f}")
    print(f"\nFinal training loss (average over the run): {train_result.training_loss:.4f}")
    print(f"Training runtime: {train_result.metrics['train_runtime']:.1f} s")


print_training_log(trainer_r4, train_result_r4)

### Saving the Adapter

Like in the demo, save_pretrained on a PEFT model only writes the LoRA adapter weights and the adapter config, not the full base model.

In [ ]:
lora_model_r4.save_pretrained(ADAPTER_DIR_R4)

print("Saved adapter to", ADAPTER_DIR_R4)
for f in sorted(os.listdir(ADAPTER_DIR_R4)):
    print(f"  {f:<30} {os.path.getsize(os.path.join(ADAPTER_DIR_R4, f)) / 1024:.1f} KB")

## Task 5: Post Fine-Tuning Inference

### Reloading the Saved Adapter

To make sure the saved adapter actually works on its own, I load a fresh copy of flan-t5-small, attach the adapter from disk with PeftModel.from_pretrained, and then merge it into the base weights with merge_and_unload, the same way the demo does. The fresh base model matters here: get_peft_model injected the r=4 layers into the model object used for training, so reloading into that same object would end up with two adapters on it.

In [ ]:
def load_finetuned_model(adapter_dir):
    fresh_base = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model = PeftModel.from_pretrained(fresh_base, adapter_dir)  # base model + trained LoRA adapter
    model = model.merge_and_unload()  # fold the LoRA update into the base weights
    return model.to(device).eval()


finetuned_r4 = load_finetuned_model(ADAPTER_DIR_R4)
print("LoRA model loaded from", ADAPTER_DIR_R4)

### Summaries After Fine-Tuning

These are the same two dialogues as the baseline, run through the same generate_summary function with the same GEN_KWARGS.

In [ ]:
lora_r4_outputs = []
for ex, base_out in zip(eval_examples, baseline_outputs):
    lora_summary = generate_summary(finetuned_r4, ex["input"])
    lora_r4_outputs.append({"example": ex["example"], "id": ex["id"], "lora_r4": lora_summary})

    print(f"=============== Example {ex['example']} (test index {ex['test_index']}, id {ex['id']}) ===============")
    print("DIALOGUE:")
    print(ex["dialogue"])
    print()
    print("REFERENCE SUMMARY:")
    print(ex["reference"])
    print()
    print("BEFORE FINE-TUNING:")
    print(base_out["baseline"])
    print()
    print("AFTER FINE-TUNING (LoRA r=4):")
    print(lora_summary)
    print()

## Task 6: Output Comparison

In [ ]:
pd.set_option("display.max_colwidth", None)

# Observations written after reading the outputs printed above
observations_r4 = [
    "Baseline just asks a question instead of summarizing. The LoRA output is in summary form and gets the band plan and "
    "Vanilla Ice (hip hop) right, but mislabels the speakers as 'Ernie and Ernie' and copies a sentence almost word for word.",
    "Baseline again copies a question from the dialogue. The LoRA output uses the #Person# format, but the facts are wrong: "
    "they are already in New Orleans, the jazz club and riverboat were turned down, and the theater decision is missing.",
]

comparison_df = pd.DataFrame({
    "Example": [ex["example"] for ex in eval_examples],
    "Reference summary": [ex["reference"] for ex in eval_examples],
    "Baseline output": [b["baseline"] for b in baseline_outputs],
    "LoRA output (r=4)": [l["lora_r4"] for l in lora_r4_outputs],
    "Observation": observations_r4,
})
comparison_df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index")

### What Changed

The clearest change is the format. Before fine-tuning, flan-t5-small did not really summarize either dialogue and just returned a question lifted from the conversation ("What's the name of the band?", "How about a riverboat tour?"). After LoRA fine-tuning, both outputs read like DialogSum summaries, written in the third person and using the #Person1#/#Person2# tags from the training data.

The content is only partly correct though. Example 1 captures the main idea (starting a band and playing Vanilla Ice songs) but calls both speakers "Ernie" and is longer than the reference. Example 2 is still wrong on the key facts, since it says the speaker wants the two activities they actually rejected and never mentions the theater, which is the whole point of the reference summary. So 250 steps of LoRA training taught the model the style of a summary more than it improved its understanding of what happened in the conversation.

## Task 7: LoRA Rank Experiment (r=4 vs r=16)

### Training the r=16 Adapter

The second configuration uses r=16, lora_alpha=32, and lora_dropout=0.1. Everything else stays the same as the r=4 run: the base model, tokenizer, 1,000-example training subset, preprocessing, SEED, 2 epochs, batch size of 8, learning rate of 1e-3, and GEN_KWARGS. That way the rank (along with alpha, which the assignment pairs with it) is the only thing that changes.

One thing worth pointing out is that the effective scaling alpha / r is actually lower for this run: 32 / 16 = 2, compared to 16 / 4 = 4 for r=4. So r=16 has more capacity, but each unit of its update is scaled down more.

In [ ]:
LORA_R16 = {"r": 16, "lora_alpha": 32, "lora_dropout": 0.1}
ADAPTER_DIR_R16 = "outputs/dialogsum_lora_r16"

lora_model_r16 = build_lora_model(**LORA_R16)
params_r16 = print_trainable_parameters(lora_model_r16)

In [ ]:
trainer_r16, train_result_r16 = train_lora(lora_model_r16, ADAPTER_DIR_R16)

In [ ]:
print_training_log(trainer_r16, train_result_r16)

lora_model_r16.save_pretrained(ADAPTER_DIR_R16)
print("\nSaved adapter to", ADAPTER_DIR_R16)

### Confirming Both Adapters Are Saved Separately

Each directory should have its own adapter_config.json with the right rank, and the r=16 weights file should be about four times larger than the r=4 one.

In [ ]:
import json

for adapter_dir in [ADAPTER_DIR_R4, ADAPTER_DIR_R16]:
    with open(os.path.join(adapter_dir, "adapter_config.json")) as f:
        cfg = json.load(f)
    weights_kb = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1024
    print(f"{adapter_dir:<30} r={cfg['r']:<3} lora_alpha={cfg['lora_alpha']:<3} "
          f"target_modules={sorted(cfg['target_modules'])} weights={weights_kb:.1f} KB")

### Inference with the r=16 Adapter

Same reload process as Task 5 (a fresh base model, then the adapter loaded from disk and merged in), and the same two dialogues with the same GEN_KWARGS.

In [ ]:
finetuned_r16 = load_finetuned_model(ADAPTER_DIR_R16)

lora_r16_outputs = []
for ex, r4_out in zip(eval_examples, lora_r4_outputs):
    r16_summary = generate_summary(finetuned_r16, ex["input"])
    lora_r16_outputs.append({"example": ex["example"], "id": ex["id"], "lora_r16": r16_summary})

    print(f"=============== Example {ex['example']} (test index {ex['test_index']}, id {ex['id']}) ===============")
    print("REFERENCE SUMMARY:")
    print(ex["reference"])
    print()
    print("LoRA r=4 :", r4_out["lora_r4"])
    print("LoRA r=16:", r16_summary)
    print()

### Rank Comparison Table

In [ ]:
rank_rows = []
for label, cfg, params, outputs, key, result in [
    ("r=4", LORA_R4, params_r4, lora_r4_outputs, "lora_r4", train_result_r4),
    ("r=16", LORA_R16, params_r16, lora_r16_outputs, "lora_r16", train_result_r16),
]:
    rank_rows.append({
        "Rank": cfg["r"],
        "lora_alpha": cfg["lora_alpha"],
        "Trainable params": f"{params['trainable']:,}",
        "Trainable %": f"{params['pct']:.4f}%",
        "Final train loss": round(result.training_loss, 4),
        "Example 1 output": outputs[0][key],
        "Example 2 output": outputs[1][key],
    })

rank_df = pd.DataFrame(rank_rows)
rank_df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index")

### Effect of Rank

Going from r=4 to r=16 multiplies the trainable parameters by four (172,032 to 688,128, or 0.22% to 0.89% of the model), and the r=16 run ended with a slightly lower average training loss (1.6007 vs 1.6292). That extra capacity did not really show up in the generated summaries, though. For Example 1 the two outputs are nearly word for word the same, with the only difference being "They'll" instead of "They're going to". For Example 2, r=16 mentions both speakers instead of just #Person2#, but it makes the same factual mistakes as r=4 (going "to New Orleans", the jazz club and riverboat tour, no theater).

Based on what I see here, rank was not the limiting factor for this setup. Both adapters learned the same summary style and made the same mistakes, which points more to the small base model and the short training budget than to adapter capacity. With only two examples I cannot say r=16 is better or worse in general, just that at this budget the difference between the two ranks is small enough that r=4 gets about the same result with a quarter of the trainable parameters.

## Final Observations

- LoRA on the q and v projections of flan-t5-small trains only 0.22% of the parameters at r=4 (0.89% at r=16), and each saved adapter is under 3 MB, compared to roughly 300 MB for the full model in fp32. Both adapters were saved separately and reloaded from disk onto a fresh base model before inference.
- The training loss dropped from about 2.1 in the first logging window to about 1.5 by the end of the second epoch for both ranks, so the adapters did learn from the 1,000-example subset.
- The most visible effect of fine-tuning was the output format. The base model returned a question copied from the dialogue for both examples, while the fine-tuned models produced proper third-person summaries in the DialogSum style.
- The summaries are still imperfect: wrong speaker names in Example 1 and wrong facts in Example 2 for both ranks. r=16 did not fix these, so for this model size and training budget, increasing the rank gave little visible benefit.